# 04 · Robustness, errors and ablations
`metrics.json` (robustness), `errors.json` and `ablations.json` (Δ vs the full system with a paired group bootstrap, 90 % interval).

In [1]:
import json
from pathlib import Path

import pandas as pd

R = Path("../reports")  # the only directory read (scripts/run_notebooks.py --check-reads-only)


def load(name):
    return json.loads((R / name).read_text(encoding="utf-8"))
m = load("metrics.json")
print(json.dumps(m.get("robustness", {}), indent=1)[:1500])

{
 "note": "data/robustness/probes.csv does not exist yet (P2.5 is a human task); per-style scores appear once it is written",
 "status": "skipped"
}


In [2]:
err = load("errors.json")
e = pd.DataFrame(err["our_test"])
if len(e):
    e["gold_cat"] = e.gold.str.split(" / ").str[0]
    e["pred_cat"] = e.pred.str.split(" / ").str[0]
    display(e.groupby(["gold_cat", "pred_cat", "lang"]).size().sort_values(ascending=False).head(15))

gold_cat          pred_cat          lang   
Transportation    Transportation    ar         12
                                    en         11
Facilities        Facilities        ar          9
Finance           Registration      arabizi     8
Facilities        Facilities        en          7
                  Academic Support  arabizi     6
Academic Support  Registration      ar          6
IT Support        IT Support        en          6
Transportation    Transportation    arabizi     6
Registration      Registration      en          5
Academic Support  Registration      arabizi     5
Finance           Academic Support  arabizi     5
Facilities        IT Support        en          4
                  Registration      arabizi     4
Academic Support  Academic Support  ar          4
dtype: int64

In [3]:
abl = load("ablations.json")
rows = []
for arm, a in abl["arms"].items():
    for target in ("category", "priority"):
        o = a.get("oof_official", {}).get(target, {})
        t = a.get("our_test", {}).get(target, {})
        rows.append({"arm": arm, "target": target, "oof": o.get("macro_f1"), "oof_delta": o.get("delta"),
                     "oof_ci90": o.get("ci90"), "our_test": t.get("macro_f1"),
                     "our_test_delta": t.get("delta")})
display(pd.DataFrame(rows).set_index(["arm", "target"]))

oof  oof_delta            oof_ci90  \
arm                      target                                            
+knn_in_category         category  0.9351     0.0000          [0.0, 0.0]   
                         priority  0.9036     0.0000          [0.0, 0.0]   
+m1_in_category          category  0.9351     0.0000          [0.0, 0.0]   
                         priority  0.9036     0.0000          [0.0, 0.0]   
+unreviewed_augmentation category  0.9794     0.0443       [0.0, 0.1031]   
                         priority  0.9265     0.0229       [0.0, 0.0717]   
-augmentation            category  0.9351     0.0000          [0.0, 0.0]   
                         priority  0.9036     0.0000          [0.0, 0.0]   
-m1                      category  0.9351     0.0000          [0.0, 0.0]   
                         priority  0.9036     0.0000          [0.0, 0.0]   
-m2knn                   category  0.9351     0.0000          [0.0, 0.0]   
                         priority  0.9036     0.0000          [0.0, 0.0]   
-m2lr                    category  0.9351     0.0000          [0.0, 0.0]   
                         priority  0.9036     0.0000          [0.0, 0.0]   
-m2proto                 category  0.6130    -0.3221  [-0.4515, -0.2308]   
                         priority  0.9036     0.0000          [0.0, 0.0]   
-urgency_features        category  0.9351     0.0000          [0.0, 0.0]   
                         priority  0.6147    -0.2889  [-0.4334, -0.1602]   
-wrapper_stripping       category  0.8669    -0.0682  [-0.1182, -0.0383]   
                         priority  0.9036     0.0000          [0.0, 0.0]   
full                     category  0.9351     0.0000          [0.0, 0.0]   
                         priority  0.9036     0.0000          [0.0, 0.0]   
hard_targets             category  0.9351     0.0000          [0.0, 0.0]   
                         priority  0.9036     0.0000          [0.0, 0.0]   
mask_deadline            category  0.9351     0.0000          [0.0, 0.0]   
                         priority  0.9036     0.0000          [0.0, 0.0]   
proto_learned_only       category  0.6059    -0.3292  [-0.4545, -0.2299]   
                         priority  0.9036     0.0000          [0.0, 0.0]   
proto_office_only        category  0.8641    -0.0709      [-0.1567, 0.0]   
                         priority  0.9036     0.0000          [0.0, 0.0]   
tuned_stage_b            category  0.9391     0.0041   [-0.0444, 0.0545]   
                         priority  0.9036     0.0000          [0.0, 0.0]   

                                   our_test  our_test_delta  
arm                      target                              
+knn_in_category         category    0.7697          0.0007  
                         priority    0.7521          0.0000  
+m1_in_category          category    0.7862          0.0172  
                         priority    0.7521          0.0000  
+unreviewed_augmentation category    0.8084          0.0393  
                         priority    0.8365          0.0845  
-augmentation            category    0.7691          0.0000  
                         priority    0.7521          0.0000  
-m1                      category    0.7691          0.0000  
                         priority    0.7521          0.0000  
-m2knn                   category    0.7691          0.0000  
                         priority    0.7521          0.0000  
-m2lr                    category    0.7669         -0.0021  
                         priority    0.7521          0.0000  
-m2proto                 category    0.7658         -0.0033  
                         priority    0.7521          0.0000  
-urgency_features        category    0.7691          0.0000  
                         priority    0.6243         -0.1277  
-wrapper_stripping       category    0.7872          0.0181  
                         priority    0.6361         -0.1160  
full                     category    0.7691          0.0000  
                         pr